In [1]:
import tensorflow as tf
from tensorflow.keras import layers, models, Model, Input
import pandas as pd

In [2]:
personality_dim = pd.read_csv("perso.csv").shape[1]
trip_dim = pd.read_csv("trip.csv").shape[1]
num_categories = pd.read_csv("categories.csv").shape[1]

BATCH_SIZE = 32
EPOCHS = 1

In [3]:
X_person = pd.read_csv("perso.csv").astype("float32").values
X_trip   = pd.read_csv("trip.csv").astype("float32").values
y_vals   = pd.read_csv("categories.csv").astype("float32").values

In [4]:
dataset = tf.data.Dataset.from_tensor_slices(
    ((X_person, X_trip), y_vals)
)

train_ds = (
    dataset.shuffle(5000)
           .batch(BATCH_SIZE)
           .prefetch(tf.data.AUTOTUNE)
)

In [5]:
dense_p1 = layers.Dense(16, activation="relu", name="p_dense_1")
bn_p1    = layers.BatchNormalization(name="p_bn_1")
dense_p2 = layers.Dense(8, activation="relu", name="p_dense_2")
personality_embed_layer = layers.Dense(4, activation="relu", name="personality_embedding")

In [6]:
dense_t1 = layers.Dense(16, activation="relu", name="t_dense_1")
bn_t1    = layers.BatchNormalization(name="t_bn_1")
dense_t2 = layers.Dense(8, activation="relu", name="trip_features")

In [7]:
fusion_dense1 = layers.Dense(32, activation="relu", name="fusion_dense_1")
fusion_dense2 = layers.Dense(16, activation="relu", name="fusion_dense_2")

In [8]:
category_output_layer = layers.Dense(num_categories, activation="sigmoid", name="category_output")

In [9]:
personality_input = Input(shape=(personality_dim,), name="personality_input")
trip_input = Input(shape=(trip_dim,), name="trip_input")

In [10]:
p = dense_p1(personality_input)
p = bn_p1(p)
p = dense_p2(p)
personality_embedding = personality_embed_layer(p)
t = dense_t1(trip_input)
t = bn_t1(t)
t = dense_t2(t)
merged = layers.Concatenate(name="fusion_layer")([personality_embedding, t])
m = fusion_dense1(merged)
m = fusion_dense2(m)
category_output = category_output_layer(m)

In [11]:
model = Model(
    inputs=[personality_input, trip_input],
    outputs=[category_output]
)

In [12]:
model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=[
    tf.keras.metrics.BinaryAccuracy(),
    tf.keras.metrics.AUC(multi_label=True)
]
)


In [13]:
history = model.fit(train_ds, epochs=EPOCHS)

2/2 [==============================] - 6s 17ms/step - loss: 0.6999 - binary_accuracy: 0.5067 - auc: 0.5011


In [14]:
model.save("travel_ann_full.h5")


In [15]:
personality_encoder = Model(
    inputs=personality_input,
    outputs=personality_embedding,
    name="personality_encoder"
)

personality_encoder.save("personality_encoder.h5")
